<!-- track-identity-card -->
# Behavioral fingerprint over 19 metrics

| | |
|---|---|
| Pipeline step | `03_fingerprint.ipynb` |
| Manuscript section | 3.3 |
| Copied from | `notebooks/NB09_fingerprint_v3.ipynb` |
| Source sha256 | `feebca88326634c915222fdeb622e456` |

**Reads**

- `data/features/driver_corner_matrix_<track>.parquet`

**Writes**

- `data/fingerprints/fingerprint_cross_track.parquet`
- `data/fingerprints/cluster_profiles.json`
- `data/fingerprints/v3_info.json`

Builds the four-dimensional fingerprint described in Section 3.3.

> Copied verbatim from the working notebook. The identity card above is the only addition; no code cell was modified.


# 09 — Driver Style Profiling v3: 19-Metrikli Fingerprint
**Sim Racing Telemetry Analysis — MSc Thesis**

## v2 → v3 Değişiklikleri

| Konu | v2 | v3 |
|------|----|----|
| Metrik sayısı | 9 (2+3+1+3) | **19 (5+6+4+4)** |
| Yeni B1 metrikleri | — | mean_mc_speed_ratio, mean_mc_lateral, mean_cex_accel_rate |
| Yeni B2 metrikleri | — | mean_slb_dist, mean_slb_decel, mean_ce_brake_turnin |
| Yeni B3 metrikleri | — | mean_cex_throttle_lag, pct_lift_coast, pct_flat_out |
| Yeni B4 metriği | — | braking_dist_std |
| Clipping | Yok | mean_slb_dist negatif → 0 |
| Before/after | Yok | Eski vs yeni Silhouette karşılaştırma |
| NaN filtre | ✅ Korundu | ✅ Aynen devam |
| Hibrit referans | ✅ Korundu | ✅ Aynen devam |
| Cluster etiketleme | ✅ Korundu | ✅ Aynen devam |


In [ ]:
# track-config-bootstrap
# Locates track/config.py, which resolves the data root at run time.
# Works from a flat layout (track/ beside the notebooks) and from the
# repository layout (src/track/ one level up). See track/config.py.
import sys as _sys, pathlib as _pl
_cands = []
for _p in [_pl.Path.cwd()] + list(_pl.Path.cwd().parents):
    _cands += [_p, _p / "src"]
for _c in _cands:
    if (_c / "track" / "config.py").is_file():
        _sys.path.insert(0, str(_c))
        break
else:
    raise RuntimeError(
        "track/config.py not found. Run this notebook from inside the repository, "
        "or add the directory holding track/ to sys.path."
    )
from track.config import PROJECT_ROOT as TRACK_ROOT
print("data root:", TRACK_ROOT)


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

PROJECT      = TRACK_ROOT
FEATURES_DIR = PROJECT / "data" / "features"
FP_DIR       = PROJECT / "data" / "fingerprints"
FIG_DIR      = PROJECT / "results" / "figures"
FP_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

TRACKS = ['monza', 'barcelona', 'red_bull_ring']

# ══════════════════════════════════════════════════════════════
# 19-METRİKLİ BOYUT HARİTASI (v3)
# ══════════════════════════════════════════════════════════════
# Eski v2: 9 metrik (2+3+1+3)
DIMENSIONS_OLD = {
    'B1_Hiz':       {'label': 'Hiz Yonetimi',      'metrics': ['mean_apex_speed', 'speed_loss_eff']},
    'B2_Frenleme':  {'label': 'Frenleme Stili',     'metrics': ['mean_brake_pressure', 'trail_braking_ratio', 'mean_trail_pressure']},
    'B3_Strateji':  {'label': 'Suruc Stratejisi',   'metrics': ['mean_coasting_dist']},
    'B4_Tutarlilik':{'label': 'Tutarlilik',         'metrics': ['apex_speed_std', 'exit_speed_std', 'speed_loss_eff_std']},
}

# Yeni v3: 19 metrik (5+6+4+4)
DIMENSIONS = {
    'B1_Hiz': {
        'label': 'Hız Yönetimi',
        'metrics': [
            'mean_apex_speed',        # Viraj ortası hız seviyesi
            'speed_loss_eff',          # Hız kaybı verimliliği (apex/entry)
            'mean_mc_speed_ratio',     # Mid-corner hız oranı (MC/entry)
            'mean_mc_lateral',         # Mid-corner yanal ivme (g)
            'mean_cex_accel_rate',     # Çıkış ivmesi (km/h per meter)
        ]
    },
    'B2_Frenleme': {
        'label': 'Frenleme Stili',
        'metrics': [
            'mean_brake_pressure',     # Ortalama fren basıncı
            'trail_braking_ratio',     # Trail braking oranı
            'mean_trail_pressure',     # Trail braking basınç seviyesi
            'mean_slb_dist',           # Düz fren mesafesi (SLB fazı)
            'mean_slb_decel',          # SLB yavaşlama oranı
            'mean_ce_brake_turnin',    # Turn-in'de fren durumu (0/1)
        ]
    },
    'B3_Strateji': {
        'label': 'Sürüş Stratejisi',
        'metrics': [
            'mean_coasting_dist',      # Coasting (ne gaz ne fren) mesafesi
            'mean_cex_throttle_lag',   # Apex→gaz gecikmesi (metre)
            'pct_lift_coast',          # Lift & coast viraj yüzdesi
            'pct_flat_out',            # Tam gaz viraj yüzdesi
        ]
    },
    'B4_Tutarlilik': {
        'label': 'Tutarlılık',
        'metrics': [
            'apex_speed_std',          # Apex hızı standart sapması
            'exit_speed_std',          # Çıkış hızı standart sapması
            'speed_loss_eff_std',      # Hız kaybı verimliliği std
            'braking_dist_std',        # Fren mesafesi standart sapması
        ]
    },
}
DIM_NAMES = list(DIMENSIONS.keys())

# Hibrit referans parametreleri
TOP_N = 2
CLOSE_THRESHOLD = 0.05

n_old = sum(len(d['metrics']) for d in DIMENSIONS_OLD.values())
n_new = sum(len(d['metrics']) for d in DIMENSIONS.values())
print(f"✅ DIMENSIONS güncellendi: {n_old} metrik (v2) → {n_new} metrik (v3)")
print(f"   B1: {len(DIMENSIONS['B1_Hiz']['metrics'])} metrik")
print(f"   B2: {len(DIMENSIONS['B2_Frenleme']['metrics'])} metrik")
print(f"   B3: {len(DIMENSIONS['B3_Strateji']['metrics'])} metrik")
print(f"   B4: {len(DIMENSIONS['B4_Tutarlilik']['metrics'])} metrik")


## Adım 1 — Matrix Yükleme + NaN Filtre + Clipping

Üç fix uygulanıyor:
1. `mean_apex_speed` NaN veya `n_corners_valid == 0` → sürücü çıkarılır
2. `mean_slb_dist` negatif değerler → 0'a cliplenir (Monza'da -460 gibi değerler var)
3. Tüm metrik mevcudiyet kontrolü


In [ ]:
matrices = {}
n_filtered_total = 0
clip_counts = {}

ALL_METRICS = []
for cfg in DIMENSIONS.values():
    ALL_METRICS.extend(cfg['metrics'])

for track in TRACKS:
    mat_path = FEATURES_DIR / f"driver_corner_matrix_{track}.parquet"
    if not mat_path.exists():
        print(f"❌ {track}: matrix bulunamadı — atlanıyor")
        continue
    
    mat = pd.read_parquet(mat_path)
    n_before = len(mat)
    
    # Fix 1: NaN filtre
    valid_mask = (
        mat['mean_apex_speed'].notna() & 
        (mat['n_corners_valid'] > 0)
    )
    mat = mat[valid_mask].reset_index(drop=True)
    n_after = len(mat)
    n_filtered = n_before - n_after
    n_filtered_total += n_filtered
    
    # Fix 2: mean_slb_dist negatif clipping
    if 'mean_slb_dist' in mat.columns:
        neg_count = (mat['mean_slb_dist'] < 0).sum()
        if neg_count > 0:
            mat['mean_slb_dist'] = mat['mean_slb_dist'].clip(lower=0)
            clip_counts[track] = neg_count
    
    # Metrik mevcudiyet kontrolü
    available = [m for m in ALL_METRICS if m in mat.columns]
    missing   = [m for m in ALL_METRICS if m not in mat.columns]
    
    matrices[track] = mat
    
    status = f"({n_filtered} filtrelendi)" if n_filtered > 0 else ""
    clip_status = f"({clip_counts.get(track, 0)} negatif cliplendi)" if track in clip_counts else ""
    print(f"✅ {track}: {n_after} sürücü {status} {clip_status}")
    print(f"   Mevcut metrikler: {len(available)}/{len(ALL_METRICS)}")
    if missing:
        print(f"   ⚠️ Eksik: {missing}")

print(f"\nToplam filtrelenen: {n_filtered_total} sürücü")
print(f"Toplam cliplenen: {sum(clip_counts.values())} kayıt")


## Adım 2 — Pist Bazlı Fingerprint Hesaplama

Her sürücü için 4 boyut skoru hesaplanır:
- Boyut skoru = o boyuttaki metriklerin normalize ortalaması
- Normalizasyon: pist-bazlı min-max [0, 1]
- B4 (Tutarlılık): ters çevrilir (yüksek std = düşük tutarlılık)


In [ ]:
def compute_fingerprint_track(df_track, track_name):
    rows = []
    for _, drv in df_track.iterrows():
        row = {'driver_id': drv['driver_id'], 'track': track_name}
        for dim, cfg in DIMENSIONS.items():
            vals = []
            for m in cfg['metrics']:
                if m in drv.index and pd.notna(drv[m]):
                    vals.append(float(drv[m]))
            row[dim] = np.mean(vals) if vals else np.nan
        rows.append(row)
    
    fp = pd.DataFrame(rows)
    
    # Pist-bazlı min-max normalizasyon
    for dim in DIM_NAMES:
        col = fp[dim].copy()
        if dim == 'B4_Tutarlilik':
            col = 1.0 - col  # Ters: yüksek std = düşük tutarlılık
        mn, mx = col.min(), col.max()
        if mx > mn:
            fp[dim] = (col - mn) / (mx - mn)
        else:
            fp[dim] = 0.5
    
    return fp

# Hesapla
fps_track = {}
for track in TRACKS:
    if track not in matrices:
        continue
    fps_track[track] = compute_fingerprint_track(matrices[track], track)
    fp = fps_track[track]
    scores = ' '.join(f"B{i+1}:{fp[d].mean():.3f}" for i, d in enumerate(DIM_NAMES))
    print(f"  [{track}] {len(fp)} sürücü | {scores}")


## Adım 3 — Cross-Track Fingerprint (Viraj-Sayısı Ağırlıklı)

In [ ]:
driver_sets = {t: set(fps_track[t]['driver_id']) for t in fps_track}
common_3 = sorted(set.intersection(*driver_sets.values()))
print(f"3 pistte ortak: {len(common_3)} sürücü")

def compute_cross_track_fp(fps_track, driver_ids):
    rows = []
    for did in driver_ids:
        row = {'driver_id': did, 'n_tracks': 0}
        dim_vals = {d: [] for d in DIM_NAMES}
        for track, fp in fps_track.items():
            drv = fp[fp['driver_id'] == did]
            if len(drv) == 0:
                continue
            n_valid = 1
            if track in matrices:
                m = matrices[track]
                m_drv = m[m['driver_id'] == did]
                if len(m_drv) > 0:
                    n_valid = int(m_drv.iloc[0].get('n_corners_valid', 1))
            for d in DIM_NAMES:
                val = drv.iloc[0][d]
                if pd.notna(val):
                    dim_vals[d].append((val, n_valid))
            row['n_tracks'] += 1
        
        for d in DIM_NAMES:
            if dim_vals[d]:
                vals, wts = zip(*dim_vals[d])
                row[d] = float(np.average(vals, weights=wts))
            else:
                row[d] = np.nan
        rows.append(row)
    return pd.DataFrame(rows)

fp_cross = compute_cross_track_fp(fps_track, common_3)
print(f"\nCross-track fingerprint ({len(fp_cross)} sürücü):")
for _, r in fp_cross.sort_values('B1_Hiz', ascending=False).iterrows():
    bars = '  '.join(f"{r[d]:.3f}" for d in DIM_NAMES)
    print(f"  {r['driver_id']:<25s} {bars}")


## Adım 4 — K-Means Kümeleme

In [ ]:
X = fp_cross[DIM_NAMES].fillna(0.5).values

print("Silhouette analizi (v3 — 19 metrik):")
sil_results = {}
for k in range(2, min(len(fp_cross), 6)):
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(X)
    sil = silhouette_score(X, labels)
    sizes = pd.Series(labels).value_counts().sort_index().tolist()
    sil_results[k] = sil
    print(f"  k={k}: silhouette={sil:.3f}  boyutlar={sizes}")

# k=3 zorlama (tez anlatısı için)
km3 = KMeans(n_clusters=3, random_state=42, n_init=20)
fp_cross['cluster'] = km3.fit_predict(X)

print(f"\nk=3 ile cluster merkezleri:")
for ci in range(3):
    members = fp_cross[fp_cross['cluster'] == ci]
    center = {d: members[d].mean() for d in DIM_NAMES}
    vals = '  '.join(f"{d.split('_')[0]}={center[d]:.3f}" for d in DIM_NAMES)
    print(f"  Cluster {ci}: {vals}")
    
print(f"\nSürücü dağılımı:")
for ci in range(3):
    members = fp_cross[fp_cross['cluster'] == ci]['driver_id'].tolist()
    print(f"  Cluster {ci}: {members}")


## Adım 5 — Before/After Karşılaştırma

v2 (9 metrik) vs v3 (19 metrik) Silhouette karşılaştırması.


In [ ]:
# v2 fingerprint'i eski metriklerle hesapla (karşılaştırma için)
def compute_fp_with_dims(dim_map, matrices, tracks, common_ids):
    fps_t = {}
    for track in tracks:
        if track not in matrices:
            continue
        rows = []
        for _, drv in matrices[track].iterrows():
            row = {'driver_id': drv['driver_id'], 'track': track}
            for dim, cfg in dim_map.items():
                vals = [float(drv[m]) for m in cfg['metrics'] 
                        if m in drv.index and pd.notna(drv[m])]
                row[dim] = np.mean(vals) if vals else np.nan
            rows.append(row)
        fp = pd.DataFrame(rows)
        dim_names = list(dim_map.keys())
        for dim in dim_names:
            col = fp[dim].copy()
            if 'Tutarlilik' in dim:
                col = 1.0 - col
            mn, mx = col.min(), col.max()
            if mx > mn:
                fp[dim] = (col - mn) / (mx - mn)
            else:
                fp[dim] = 0.5
        fps_t[track] = fp
    
    # Cross-track
    rows = []
    dim_names = list(dim_map.keys())
    for did in common_ids:
        row = {'driver_id': did}
        for d in dim_names:
            vals = []
            for t, fp in fps_t.items():
                drv = fp[fp['driver_id'] == did]
                if len(drv) > 0 and pd.notna(drv.iloc[0][d]):
                    vals.append(drv.iloc[0][d])
            row[d] = np.mean(vals) if vals else np.nan
        rows.append(row)
    return pd.DataFrame(rows)

# v2 hesapla
fp_old = compute_fp_with_dims(DIMENSIONS_OLD, matrices, TRACKS, common_3)
X_old = fp_old[list(DIMENSIONS_OLD.keys())].fillna(0.5).values
km_old = KMeans(n_clusters=3, random_state=42, n_init=20)
labels_old = km_old.fit_predict(X_old)
sil_old = silhouette_score(X_old, labels_old)

# v3 (zaten hesaplandı)
sil_new = silhouette_score(X, fp_cross['cluster'].values)

print("╔══════════════════════════════════════════════════╗")
print("║        BEFORE / AFTER KARŞILAŞTIRMA             ║")
print("╠══════════════════════════════════════════════════╣")
print(f"║  Metrik sayısı   :  {n_old:2d} (v2)  →  {n_new:2d} (v3)         ║")
print(f"║  Silhouette (k=3):  {sil_old:.3f} (v2)  →  {sil_new:.3f} (v3)      ║")
delta = sil_new - sil_old
sign = "+" if delta >= 0 else ""
print(f"║  Değişim          :  {sign}{delta:.3f}                       ║")
print(f"║  Cross-track      :  {len(common_3)} sürücü (değişmedi)       ║")
print("╚══════════════════════════════════════════════════╝")

# Cluster değişim analizi
print("\nCluster atama değişimleri:")
for i, did in enumerate(common_3):
    old_c = labels_old[i]
    new_c = fp_cross[fp_cross['driver_id'] == did]['cluster'].iloc[0]
    changed = " ← DEĞİŞTİ" if old_c != new_c else ""
    print(f"  {did:<25s} v2:C{old_c} → v3:C{new_c}{changed}")


## Adım 6 — Cluster Profilleri (Otomatik Etiketleme)

In [ ]:
LABEL_MAP = {
    'B1_Hiz': 'Hız Odaklı',
    'B2_Frenleme': 'Agresif Frenleyici',
    'B3_Strateji': 'Coasting Odaklı',
    'B4_Tutarlilik': 'Tutarlı Sürücü',
}

BALANCED_THRESHOLD = 0.55

CLUSTER_PROFILES = {}
for ci in range(3):
    members = fp_cross[fp_cross['cluster'] == ci]
    center = {d: members[d].mean() for d in DIM_NAMES}
    dominant = max(DIM_NAMES, key=lambda d: center[d])
    
    if center[dominant] < BALANCED_THRESHOLD:
        label = 'Dengeli Sürücü'
    else:
        label = LABEL_MAP[dominant]
    
    CLUSTER_PROFILES[ci] = {
        'label': label,
        'center': center,
        'dominant': dominant,
        'members': members['driver_id'].tolist(),
    }
    print(f"  Cluster {ci} — '{label}' ({len(members)} sürücü)")
    print(f"    Baskın: {DIMENSIONS[dominant]['label']} ({center[dominant]:.3f})")


## Adım 7 — Radar Chart Görselleştirme

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6), subplot_kw=dict(polar=True))
colors = ['#ff6b6b', '#4ecdc4', '#45b7d1']
labels_dim = [DIMENSIONS[d]['label'] for d in DIM_NAMES]
angles = np.linspace(0, 2 * np.pi, len(DIM_NAMES), endpoint=False).tolist()
angles += angles[:1]

for ci, ax in enumerate(axes):
    prof = CLUSTER_PROFILES[ci]
    values = [prof['center'][d] for d in DIM_NAMES]
    values += values[:1]
    
    ax.fill(angles, values, alpha=0.25, color=colors[ci])
    ax.plot(angles, values, 'o-', color=colors[ci], linewidth=2, markersize=6)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels_dim, fontsize=9)
    ax.set_ylim(0, 1)
    ax.set_title(f"C{ci}: {prof['label']}\n({len(prof['members'])} sürücü)", 
                 fontsize=12, fontweight='bold', pad=20)

plt.suptitle('Driver Fingerprint — 19 Metrikli Küme Profilleri (v3)', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'cluster_radar_v3.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Kaydedildi: {FIG_DIR / 'cluster_radar_v3.png'}")


## Adım 8 — PCA Görselleştirme

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#ff6b6b', '#4ecdc4', '#45b7d1']
for ci in range(3):
    mask = fp_cross['cluster'] == ci
    label = CLUSTER_PROFILES[ci]['label']
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], 
              c=colors[ci], s=100, label=f"C{ci}: {label}",
              edgecolors='white', linewidth=0.5, alpha=0.85)
    for idx in fp_cross[mask].index:
        ax.annotate(fp_cross.loc[idx, 'driver_id'][:10], 
                    (X_pca[idx, 0], X_pca[idx, 1]),
                    fontsize=7, ha='center', va='bottom', alpha=0.7)

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)", fontsize=11)
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)", fontsize=11)
ax.set_title('PCA — 19 Metrikli Driver Fingerprint (v3)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='best')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(FIG_DIR / 'pca_scatter_v3.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nPCA Açıklanan Varyans:")
print(f"  PC1: {pca.explained_variance_ratio_[0]*100:.1f}%")
print(f"  PC2: {pca.explained_variance_ratio_[1]*100:.1f}%")
print(f"  Toplam: {sum(pca.explained_variance_ratio_)*100:.1f}%")


## Adım 9 — Hibrit Referans Tabloları

Her viraj için cluster-bazlı top-2 + global top-2 referans.


In [ ]:
def compute_reference_tables(matrices, fp_cross):
    ref_tables = {}
    
    for track, mat in matrices.items():
        cross_ids = set(fp_cross['driver_id'])
        mat_cross = mat[mat['driver_id'].isin(cross_ids)].copy()
        
        id_to_cluster = dict(zip(fp_cross['driver_id'], fp_cross['cluster']))
        mat_cross['cluster'] = mat_cross['driver_id'].map(id_to_cluster)
        
        apex_cols = [c for c in mat.columns if c.startswith('T') and c.endswith('_apex_speed')]
        
        rows = []
        for col in apex_cols:
            corner_id = col.replace('_apex_speed', '')
            row = {'corner': corner_id, 'track': track}
            
            valid_vals = mat_cross[col].dropna()
            if len(valid_vals) >= TOP_N:
                row['global_ref'] = valid_vals.nlargest(TOP_N).mean()
            elif len(valid_vals) > 0:
                row['global_ref'] = valid_vals.max()
            else:
                row['global_ref'] = np.nan
            
            for ci in range(3):
                cluster_vals = mat_cross[mat_cross['cluster'] == ci][col].dropna()
                if len(cluster_vals) >= TOP_N:
                    row[f'cluster_{ci}_ref'] = cluster_vals.nlargest(TOP_N).mean()
                elif len(cluster_vals) > 0:
                    row[f'cluster_{ci}_ref'] = cluster_vals.max()
                else:
                    row[f'cluster_{ci}_ref'] = np.nan
            
            rows.append(row)
        
        ref_tables[track] = pd.DataFrame(rows)
    
    return ref_tables

ref_tables = compute_reference_tables(matrices, fp_cross)

print("Referans tabloları:")
for track, ref in ref_tables.items():
    print(f"\n  {track.upper()} — {len(ref)} viraj")
    for _, r in ref.iterrows():
        cluster_refs = '  '.join(f"C{i}:{r.get(f'cluster_{i}_ref', np.nan):.0f}" 
                                  for i in range(3) if pd.notna(r.get(f'cluster_{i}_ref')))
        print(f"    {r['corner']}: global={r['global_ref']:.0f}  {cluster_refs}")


## Adım 10 — Öneri Motoru v2 (Hibrit Referans)

In [ ]:
def generate_recommendations_v2(driver_id, fp_cross, matrices, ref_tables, top_n=3):
    drv_fp = fp_cross[fp_cross['driver_id'] == driver_id]
    if len(drv_fp) == 0:
        print(f"Sürücü bulunamadı: {driver_id}")
        return
    drv_fp  = drv_fp.iloc[0]
    cluster = int(drv_fp['cluster'])
    profile = CLUSTER_PROFILES[cluster]
    
    print("=" * 65)
    print(f"  SÜRÜCÜ: {driver_id}")
    print(f"  Cluster: {cluster} — '{profile['label']}'")
    print("=" * 65)
    
    # 4 Boyutlu Profil
    print("\n  4 Boyutlu Profil (19 metrik):")
    diffs = {}
    for d in DIM_NAMES:
        val     = drv_fp[d]
        ctr_val = profile['center'][d]
        diff    = val - ctr_val
        diffs[d] = diff
        bar_len = int(val * 20)
        bar = chr(9608) * bar_len + chr(9617) * (20 - bar_len)
        sign = '+' if diff > 0 else ''
        print(f"  {DIMENSIONS[d]['label']:20s} {val:.3f} {bar}  "
              f"(ort:{ctr_val:.3f}, sapma:{sign}{diff:.3f})")
    
    best  = max(diffs, key=lambda x: diffs[x])
    worst = min(diffs, key=lambda x: diffs[x])
    print(f"\n  Güçlü boyut : {DIMENSIONS[best]['label']} ({diffs[best]:+.3f})")
    print(f"  Gelişim alanı: {DIMENSIONS[worst]['label']} ({diffs[worst]:+.3f})")
    
    # Hibrit referansla viraj bazlı analiz
    corner_gaps = []
    for track, mat in matrices.items():
        drv_row = mat[mat['driver_id'] == driver_id]
        if len(drv_row) == 0 or track not in ref_tables:
            continue
        
        ref = ref_tables[track]
        apex_cols = [c for c in mat.columns if c.startswith('T') and c.endswith('_apex_speed')]
        
        for col in apex_cols:
            corner_id = col.replace('_apex_speed', '')
            drv_val = drv_row[col].iloc[0]
            if pd.isna(drv_val):
                continue
            
            ref_row = ref[ref['corner'] == corner_id]
            if len(ref_row) == 0:
                continue
            ref_row = ref_row.iloc[0]
            
            cluster_ref = ref_row.get(f'cluster_{cluster}_ref', np.nan)
            global_ref  = ref_row.get('global_ref', np.nan)
            
            if pd.isna(cluster_ref):
                continue
            
            cluster_gap = cluster_ref - drv_val
            global_gap  = global_ref - drv_val if pd.notna(global_ref) else np.nan
            
            corner_gaps.append({
                'track': track, 'corner': corner_id,
                'driver_val': drv_val, 'cluster_ref': cluster_ref,
                'global_ref': global_ref, 'cluster_gap': cluster_gap,
                'global_gap': global_gap,
            })
    
    if corner_gaps:
        sorted_gaps = sorted(corner_gaps, key=lambda x: x['cluster_gap'], reverse=True)
        improvable = [g for g in sorted_gaps if g['cluster_gap'] > 0]
        optimal    = [g for g in sorted_gaps if g['cluster_gap'] <= 0]
        
        if improvable:
            print(f"\n  📈 En fazla kazanım potansiyeli olan {min(top_n, len(improvable))} viraj:")
            for g in improvable[:top_n]:
                line = (f"    {g['track']:15s} {g['corner']}: "
                        f"senin={g['driver_val']:.0f}  "
                        f"cluster-best={g['cluster_ref']:.0f} (+{g['cluster_gap']:.0f})")
                if pd.notna(g['global_gap']) and g['global_gap'] > g['cluster_gap']:
                    line += f"  global-best={g['global_ref']:.0f} (+{g['global_gap']:.0f})"
                print(line)
        
        if optimal:
            n_show = min(2, len(optimal))
            print(f"\n  ✅ Stilin için optimal sürdüğün {n_show} viraj:")
            for g in optimal[:n_show]:
                excess = -g['cluster_gap']
                print(f"    {g['track']:15s} {g['corner']}: "
                      f"senin={g['driver_val']:.0f}  cluster-best={g['cluster_ref']:.0f} "
                      f"(+{excess:.0f} km/h üstünde!)")
    
    rec_map = {
        'B1_Hiz'       : "Apex hızını artırmak için freni biraz daha geç bırak ve mid-corner hızını koru.",
        'B2_Frenleme'  : "Trail braking tekniğiyle viraj ortası hızını artırabilirsin. SLB mesafeni kısalt.",
        'B3_Strateji'  : "Coasting mesafeni kısaltıp daha erken gaz ver. Throttle lag'i azalt.",
        'B4_Tutarlilik': "Tutarlılığı artırmak için fren noktalarını sabitle ve apex hızı varyansını azalt.",
    }
    print(f"\n  💡 Öneri: {rec_map[worst]}")


# Test: En hızlı ve en yavaş sürücü
speeds = fp_cross.sort_values('B1_Hiz', ascending=False)
print("\n" + "─" * 65)
print("  EN HIZLI SÜRÜCÜ")
generate_recommendations_v2(speeds.iloc[0]['driver_id'], fp_cross, matrices, ref_tables)

print("\n" + "─" * 65)
print("  EN YAVAŞ SÜRÜCÜ")
generate_recommendations_v2(speeds.iloc[-1]['driver_id'], fp_cross, matrices, ref_tables)


## Adım 11 — Kaydet

In [ ]:
import json

# Referans tablolarını kaydet
for track, ref in ref_tables.items():
    ref.to_parquet(FP_DIR / f'reference_{track}.parquet', index=False)

# Güncel fingerprint kaydet
fp_cross.to_parquet(FP_DIR / 'fingerprint_cross_track.parquet', index=False)

# Cluster profilleri JSON
profiles_json = {}
for ci, prof in CLUSTER_PROFILES.items():
    profiles_json[str(ci)] = {
        'label': prof['label'],
        'dominant': prof['dominant'],
        'center': {k: round(v, 4) for k, v in prof['center'].items()},
        'members': prof['members'],
    }
with open(FP_DIR / 'cluster_profiles.json', 'w') as f:
    json.dump(profiles_json, f, indent=2, ensure_ascii=False)

# v3 bilgi kaydet
v3_info = {
    'version': 'v3',
    'n_metrics': n_new,
    'n_metrics_old': n_old,
    'silhouette_v2': round(sil_old, 4),
    'silhouette_v3': round(sil_new, 4),
    'n_drivers': len(fp_cross),
    'tracks': TRACKS,
    'dimensions': {k: v['metrics'] for k, v in DIMENSIONS.items()},
}
with open(FP_DIR / 'v3_info.json', 'w') as f:
    json.dump(v3_info, f, indent=2, ensure_ascii=False)

print("Kaydedilen dosyalar:")
print(f"  fingerprint_cross_track.parquet  ({len(fp_cross)} sürücü)")
for track in ref_tables:
    print(f"  reference_{track}.parquet")
print(f"  cluster_profiles.json")
print(f"  v3_info.json")
print(f"\n✅ NB09 v3 tamamlandı — {n_new} metrik, {len(fp_cross)} sürücü")
